# Practical 4: Architectural Evolution: LeNet-5 vs. ResNet-18
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit II (Deep Architectures & Vanishing Gradients) | **CO:** CO2  
**Problem Statement Context:** Benchmarking Shallow vs. Deep Residual Networks for Crop Pattern Recognition  

---

## 1. Deep Learning Milestone Comparison
| Metric / Feature | LeNet-5 (1998) | ResNet-18 (2015) |
| :--- | :--- | :--- |
| **Pioneering Concept** | Conv + Subsampling + Tanh | Deep Residual Learning ($F(x) + x$) |
| **Typical Depth** | 5 weight layers | 18 weight layers |
| **Skip Connections** | None (Plain sequential) | Identity shortcuts around conv pairs |
| **Normalization** | None | Batch Normalization after every Conv |
| **Max Pooling vs Avg** | Average Pooling | Max Pooling + Global Average Pooling |
| **Vulnerability** | Vanishing gradient at depth > 7 | Stable gradient flow up to 1000+ layers |

In [ ]:
import sys
import time
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.models_cnn import LeNet5, ResNet18Scratch

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"[✓] Active Device: {device}")

## 2. Model Parameter Count & Architecture Inspection

In [ ]:
lenet = LeNet5(in_channels=3, num_classes=6).to(device)
resnet = ResNet18Scratch(in_channels=3, num_classes=6).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

p_lenet = count_parameters(lenet)
p_resnet = count_parameters(resnet)

print(f"LeNet-5 Trainable Parameters  : {p_lenet:,}")
print(f"ResNet-18 Trainable Parameters: {p_resnet:,}")

## 3. Training Benchmark on Synthetic Crop Foliar Subsets

In [ ]:
# Generate synthetic foliar tensor batches for benchmark training
torch.manual_seed(42)
X_dummy = torch.randn(64, 3, 32, 32)
y_dummy = torch.randint(0, 6, (64,))

dataset = TensorDataset(X_dummy, y_dummy)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

def benchmark_train(model, dataloader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for x_b, y_b in dataloader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out = model(x_b)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(dataloader))
    duration = time.time() - start_time
    return losses, duration

losses_lenet, time_lenet = benchmark_train(lenet, dataloader, epochs=8)
losses_resnet, time_resnet = benchmark_train(resnet, dataloader, epochs=8)

print(f"LeNet-5 Training Time  : {time_lenet:.2f} seconds")
print(f"ResNet-18 Training Time: {time_resnet:.2f} seconds")

## 4. Loss Convergence Curve & Parameter Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_lenet, label="LeNet-5 Loss", marker='o', color='purple')
axes[0].plot(losses_resnet, label="ResNet-18 Loss", marker='s', color='teal')
axes[0].set_title("Training Loss Convergence")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross Entropy Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(["LeNet-5", "ResNet-18"], [p_lenet, p_resnet], color=['purple', 'teal'])
axes[1].set_yscale('log')
axes[1].set_title("Model Parameter Capacity (Log Scale)")
axes[1].set_ylabel("Total Trainable Parameters")

plt.tight_layout()
plt.show()

## 5. Architectural Findings
- **Capacity:** ResNet-18 possesses vastly higher expressive capacity ($11M+$ parameters) allowing it to learn hierarchical abstractions (textures $	o$ spot shapes $	o$ disease syndromes).
- **Gradient Stability:** Skip connections preserve signal magnitude during backward passes, avoiding the severe vanishing gradients that restrict LeNet-5 to shallow networks.